# Full Dataset Preprocessing And Balancing

This notebook builds the full multilabel dataset from:

- videos: `D:/hvc/datasets/data-from-juniors/videos`
- labels: `D:/hvc/datasets/data-from-juniors/labels_final.xlsx`

It does four things:

1. Reads the full Excel label sheet and normalizes numeric `video_id` values with `str(...)`.
2. Skips missing or unreadable videos.
3. Extracts uniformly sampled frames into `D:/hvc/datasets/data-from-juniors/full_dataset_frames`.
4. Creates base and augmented manifests so training can use the full dataset with better class balance while preserving the original multilabel targets.

Augmented clips keep the same label vector and record `source_video_id`, so `new_model.ipynb` can put only original clips in validation and avoid leakage.


In [1]:
!pip install pandas

In [2]:
!pip install pillow  tqdm openpyxl 

In [3]:
from pathlib import Path
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
import math
import multiprocessing as mp
import os
import random
import warnings

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)


DATA_ROOT = Path(r"E:\m-hvc\datasets\data-from-juniors")
VIDEO_ROOT = DATA_ROOT / "videos"
LABEL_XLSX = DATA_ROOT / "labels_final.xlsx"
FRAMES_ROOT = DATA_ROOT / "full_dataset_frames"

BASE_MANIFEST_PATH = DATA_ROOT / "full_dataset_base_manifest.csv"
AUGMENTED_MANIFEST_PATH = DATA_ROOT / "full_dataset_augmented_manifest.csv"
SKIPPED_REPORT_PATH = DATA_ROOT / "full_dataset_skipped_videos.csv"
BALANCE_PLAN_PATH = DATA_ROOT / "full_dataset_balance_plan.csv"

CFG = {
    "seed": 42,
    "extract_frames_per_video": 16,
    "extract_size": 256,
    "jpeg_quality": 95,
    "overwrite_existing_frames": False,
    "num_workers": max(1, min(8, (os.cpu_count() or 4) - 1)),
    "in_flight_factor": 2,
    "save_every": 100,
    "build_augmented_variants": True,
    "overwrite_existing_augmented_frames": False,
    "max_augmented_variants_per_video": 3,
    "target_positive_quantile": 0.75,
    "min_target_positive_count": 128,
}

VALID_VIDEO_SUFFIXES = {".mp4", ".webm", ".avi", ".mov", ".mkv"}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)

print("data root:", DATA_ROOT)
print("video root:", VIDEO_ROOT)
print("label file:", LABEL_XLSX)
print("frames root:", FRAMES_ROOT)
print("cpu workers:", CFG["num_workers"])


data root: E:\m-hvc\datasets\data-from-juniors
video root: E:\m-hvc\datasets\data-from-juniors\videos
label file: E:\m-hvc\datasets\data-from-juniors\labels_final.xlsx
frames root: E:\m-hvc\datasets\data-from-juniors\full_dataset_frames
cpu workers: 8


C:\Users\Rasheek\.conda\envs\fyp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def normalize_video_id(value):
    text = str(value).strip()
    if text.endswith(".0"):
        integer_candidate = text[:-2]
        if integer_candidate.isdigit():
            text = integer_candidate
    return text


def build_video_lookup(video_root):
    extension_priority = {".mp4": 0, ".webm": 1, ".avi": 2, ".mov": 3, ".mkv": 4}
    best_files = {}

    for path in video_root.iterdir():
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        if suffix not in VALID_VIDEO_SUFFIXES:
            continue

        key = path.stem
        rank = extension_priority.get(suffix, 99)
        current = best_files.get(key)
        if current is None or rank < current[0]:
            best_files[key] = (rank, path)

    return {key: value[1] for key, value in best_files.items()}


labels_df = pd.read_excel(LABEL_XLSX, sheet_name=0, engine="openpyxl")
labels_df["source_video_id"] = labels_df["video_id"].map(normalize_video_id)

label_columns = [column for column in labels_df.columns if column not in {"video_id", "source_video_id"}]
labels_df[label_columns] = labels_df[label_columns].fillna(0).astype(int)
labels_df["label_count"] = labels_df[label_columns].sum(axis=1)
labels_df = labels_df[labels_df["label_count"] > 0].copy()

video_lookup = build_video_lookup(VIDEO_ROOT)
labels_df["video_path"] = labels_df["source_video_id"].map(lambda value: video_lookup.get(value))
labels_df["has_video_file"] = labels_df["video_path"].notna()

print(f"rows with at least one label: {len(labels_df):,}")
print(f"rows with matching video file: {int(labels_df['has_video_file'].sum()):,}")
print(f"rows missing a video file: {int((~labels_df['has_video_file']).sum()):,}")

label_summary = pd.DataFrame(
    {
        "positives": labels_df[label_columns].sum().astype(int),
        "prevalence_%": (labels_df[label_columns].mean() * 100).round(2),
    }
).sort_values("positives", ascending=False)

display(label_summary)
labels_df.head()


rows with at least one label: 5,139
rows with matching video file: 5,118
rows missing a video file: 21


,positives,prevalence_%
humour,4415,85.91
sensitive,702,13.66
anger,373,7.26
derogatory__lang,351,6.83
generic,318,6.19
positive,190,3.70
emotional,165,3.21
threat,150,2.92
political_hate,131,2.55
gender_hate,109,2.12


,video_id,generic,humour,positive,sensitive,derogatory__lang,threat,sexuality_hate,nationality_hate,caste_based_hate,...,anger,emotional,social_hate,controversial,indv_hate,gender_hate,source_video_id,label_count,video_path,has_video_file
0,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,E:\m-hvc\datasets\data-from-juniors\videos\1.mp4,True
1,2,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,2,1,E:\m-hvc\datasets\data-from-juniors\videos\2.mp4,True
2,3,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,3,1,E:\m-hvc\datasets\data-from-juniors\videos\3.mp4,True
3,4,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,4,1,E:\m-hvc\datasets\data-from-juniors\videos\4.mp4,True
4,5,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,5,1,E:\m-hvc\datasets\data-from-juniors\videos\5.mp4,True


In [5]:
def uniform_indices(frame_count, num_frames):
    if frame_count <= 1:
        return [0] * num_frames
    positions = np.linspace(0, frame_count - 1, num=num_frames)
    return np.clip(np.round(positions).astype(int), 0, frame_count - 1).tolist()


def clear_directory(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def validate_video(video_path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return {"ok": False, "reason": "open_failed"}

    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    ok, frame = capture.read()
    capture.release()

    if not ok or frame is None:
        return {"ok": False, "reason": "first_frame_failed"}
    if frame_count <= 0:
        frame_count = 1

    return {
        "ok": True,
        "reason": "ok",
        "frame_count": int(frame_count),
        "fps": fps,
        "width": width,
        "height": height,
        "duration_sec": (frame_count / fps) if fps and fps > 0 else np.nan,
    }


def extract_uniform_frames(video_path, output_dir, num_frames, image_size, jpeg_quality, overwrite=False):
    if overwrite:
        clear_directory(output_dir)
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        existing = sorted(output_dir.glob("*.jpg"))
        if len(existing) >= num_frames:
            return {"ok": True, "saved_frames": len(existing), "used_cache": True}

    video_meta = validate_video(video_path)
    if not video_meta["ok"]:
        return video_meta

    capture = cv2.VideoCapture(str(video_path))
    frame_count = video_meta["frame_count"]

    target_indices = set(uniform_indices(frame_count, num_frames))
    saved_paths = []
    current_idx = 0
    output_index = 0

    while True:
        ok, frame = capture.read()
        if not ok:
            break

        if current_idx in target_indices:
            frame = cv2.resize(frame, (image_size, image_size), interpolation=cv2.INTER_AREA)
            frame_path = output_dir / f"{output_index:04d}.jpg"

            if cv2.imwrite(str(frame_path), frame, [int(cv2.IMWRITE_JPEG_QUALITY), int(jpeg_quality)]):
                saved_paths.append(frame_path)
                output_index += 1

                if output_index >= num_frames:
                    break

        current_idx += 1

    capture.release()

    if not saved_paths:
        return {"ok": False, "reason": "no_frames_saved"}

    return {
        "ok": True,
        "reason": "ok",
        "saved_frames": len(saved_paths),
        "used_cache": False,
        **video_meta,
    }

In [6]:
from vision_classification.preprocess_cpu_mp import extract_uniform_frames_cpu_task


def persist_progress(extracted_records, skipped_records):
    pd.DataFrame(extracted_records).to_csv(BASE_MANIFEST_PATH, index=False)
    pd.DataFrame(skipped_records).to_csv(SKIPPED_REPORT_PATH, index=False)


def build_task(row):
    return {
        "source_video_id": str(row.source_video_id),
        "video_path": str(Path(row.video_path)),
        "frame_dir": str(FRAMES_ROOT / str(row.source_video_id)),
        "num_frames": CFG["extract_frames_per_video"],
        "image_size": CFG["extract_size"],
        "jpeg_quality": CFG["jpeg_quality"],
        "overwrite": CFG["overwrite_existing_frames"],
    }


candidate_df = labels_df[labels_df["has_video_file"]].copy().reset_index(drop=True)

extracted_records = []
skipped_records = []
cache_hits = 0
processed_count = 0
max_in_flight = max(CFG["num_workers"], CFG["num_workers"] * CFG["in_flight_factor"])

task_rows = [row for row in candidate_df.itertuples(index=False)]
task_iter = iter(task_rows)

progress_bar = tqdm(total=len(task_rows), desc="extract_frames_mp")
spawn_context = mp.get_context("spawn")

with ProcessPoolExecutor(
    max_workers=CFG["num_workers"],
    mp_context=spawn_context,
) as executor:
    future_to_row = {}

    while len(future_to_row) < min(max_in_flight, len(task_rows)):
        row = next(task_iter, None)
        if row is None:
            break
        future = executor.submit(extract_uniform_frames_cpu_task, build_task(row))
        future_to_row[future] = row

    while future_to_row:
        done, _ = wait(future_to_row.keys(), return_when=FIRST_COMPLETED)

        for future in done:
            row = future_to_row.pop(future)
            source_video_id = str(row.source_video_id)
            video_path = str(Path(row.video_path))
            frame_dir = str(FRAMES_ROOT / source_video_id)

            try:
                result = future.result()
            except Exception as exc:
                result = {
                    "ok": False,
                    "reason": f"worker_exception: {exc}",
                    "source_video_id": source_video_id,
                    "video_path": video_path,
                    "frame_dir": frame_dir,
                }

            if result.get("used_cache"):
                cache_hits += 1

            if not result.get("ok"):
                skipped_records.append(
                    {
                        "source_video_id": source_video_id,
                        "video_path": video_path,
                        "reason": result.get("reason", "unknown"),
                    }
                )
            else:
                record = {
                    "video_key": source_video_id,
                    "source_video_id": source_video_id,
                    "video_path": video_path,
                    "frame_dir": frame_dir,
                    "is_augmented": 0,
                    "aug_index": 0,
                    "frame_count_extracted": int(result.get("saved_frames", 0)),
                    "fps": result.get("fps", np.nan),
                    "duration_sec": result.get("duration_sec", np.nan),
                    "width": int(result.get("width", 0) or 0),
                    "height": int(result.get("height", 0) or 0),
                }
                for label in label_columns:
                    record[label] = int(getattr(row, label))
                extracted_records.append(record)

            processed_count += 1
            if processed_count % CFG["save_every"] == 0:
                persist_progress(extracted_records, skipped_records)

            progress_bar.update(1)
            progress_bar.set_postfix(
                done=len(extracted_records),
                skipped=len(skipped_records),
                cache=cache_hits,
                inflight=len(future_to_row),
            )

            next_row = next(task_iter, None)
            if next_row is not None:
                next_future = executor.submit(extract_uniform_frames_cpu_task, build_task(next_row))
                future_to_row[next_future] = next_row

progress_bar.close()

persist_progress(extracted_records, skipped_records)

base_manifest_df = pd.DataFrame(extracted_records)
skipped_df = pd.DataFrame(skipped_records)

print(f"base manifest rows: {len(base_manifest_df):,}")
print(f"skipped rows: {len(skipped_df):,}")
print(f"cache hits: {cache_hits:,}")
print("base manifest:", BASE_MANIFEST_PATH)
print("skipped report:", SKIPPED_REPORT_PATH)

display(base_manifest_df.head())
display(skipped_df.head())


extract_frames_mp: 100%|███████████████████████| 5118/5118 [1:46:57<00:00,  1.25s/it, cache=683, done=5118, inflight=0, skipped=0]

base manifest rows: 5,118
skipped rows: 0
cache hits: 683
base manifest: E:\m-hvc\datasets\data-from-juniors\full_dataset_base_manifest.csv
skipped report: E:\m-hvc\datasets\data-from-juniors\full_dataset_skipped_videos.csv


,video_key,source_video_id,video_path,frame_dir,is_augmented,aug_index,frame_count_extracted,fps,duration_sec,width,...,political_hate,religion_hate,informative,ethinity_hate,anger,emotional,social_hate,controversial,indv_hate,gender_hate
0,2,2,E:\m-hvc\datasets\data-from-juniors\videos\2.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
1,9,9,E:\m-hvc\datasets\data-from-juniors\videos\9.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
2,10,10,E:\m-hvc\datasets\data-from-juniors\videos\10.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
3,11,11,E:\m-hvc\datasets\data-from-juniors\videos\11.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,E:\m-hvc\datasets\data-from-juniors\videos\1.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0


""


In [7]:
def compute_balance_plan(base_df, label_cols, max_augmented_variants, quantile, minimum_target):
    positive_counts = base_df[label_cols].sum().astype(int)
    non_zero_counts = positive_counts[positive_counts > 0]
    target_positive_count = int(max(minimum_target, np.quantile(non_zero_counts.values, quantile)))

    per_label_extra = {}
    for label, count in positive_counts.items():
        if count <= 0:
            per_label_extra[label] = 0
            continue
        extra = int(math.ceil(target_positive_count / count) - 1)
        per_label_extra[label] = max(0, min(max_augmented_variants, extra))

    balance_rows = []
    for label in label_cols:
        balance_rows.append(
            {
                "label": label,
                "base_positives": int(positive_counts[label]),
                "target_positive_count": int(target_positive_count),
                "planned_extra_variants_for_positive_samples": int(per_label_extra[label]),
            }
        )

    balance_plan_df = pd.DataFrame(balance_rows).sort_values("base_positives", ascending=False)     
    return target_positive_count, per_label_extra, balance_plan_df


def plan_augmented_variants(base_df, label_cols, per_label_extra, max_augmented_variants):
    planned_counts = []

    for row in base_df.itertuples(index=False):
        positives = [label for label in label_cols if int(getattr(row, label)) == 1]
        if not positives:
            planned_counts.append(0)
            continue

        max_extra = max(per_label_extra[label] for label in positives)
        average_pressure = np.mean([per_label_extra[label] for label in positives])
        repeat_count = int(min(max_augmented_variants, max(max_extra, round(average_pressure))))
        planned_counts.append(repeat_count)

    planned_df = base_df[["source_video_id", "frame_dir"] + label_cols].copy()
    planned_df["planned_augmented_variants"] = planned_counts
    return planned_df


target_positive_count, per_label_extra, balance_plan_df = compute_balance_plan(
    base_manifest_df,
    label_columns,
    max_augmented_variants=CFG["max_augmented_variants_per_video"],
    quantile=CFG["target_positive_quantile"],
    minimum_target=CFG["min_target_positive_count"],
)

planned_augments_df = plan_augmented_variants(
    base_manifest_df,
    label_columns,
    per_label_extra,
    max_augmented_variants=CFG["max_augmented_variants_per_video"],
)

balance_plan_df.to_csv(BALANCE_PLAN_PATH, index=False)

print("target positive count for balancing:", target_positive_count)
print("balance plan:", BALANCE_PLAN_PATH)
display(balance_plan_df)
display(planned_augments_df[planned_augments_df["planned_augmented_variants"] > 0].head())


target positive count for balancing: 254
balance plan: E:\m-hvc\datasets\data-from-juniors\full_dataset_balance_plan.csv


,label,base_positives,target_positive_count,planned_extra_variants_for_positive_samples
1,humour,4396,254,0
3,sensitive,700,254,0
13,anger,373,254,0
4,derogatory__lang,351,254,0
0,generic,318,254,0
2,positive,190,254,1
14,emotional,164,254,1
5,threat,149,254,1
9,political_hate,131,254,1
18,gender_hate,109,254,2


,source_video_id,frame_dir,generic,humour,positive,sensitive,derogatory__lang,threat,sexuality_hate,nationality_hate,...,religion_hate,informative,ethinity_hate,anger,emotional,social_hate,controversial,indv_hate,gender_hate,planned_augmented_variants
46,51,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
59,67,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
60,68,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,3
61,69,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
71,80,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1


In [8]:
def sorted_frame_paths(frame_dir):
    frame_dir = Path(frame_dir)
    return sorted(frame_dir.glob("*.jpg"), key=lambda path: path.name)


def make_clip_augmentation_params(source_video_id, aug_index, seed):
    composite_seed = hash((str(source_video_id), int(aug_index), int(seed))) & 0xFFFFFFFF
    rng = random.Random(composite_seed)
    return {
        "crop_scale": rng.uniform(0.82, 0.98),
        "x_shift": rng.random(),
        "y_shift": rng.random(),
        "rotation": rng.uniform(-6.0, 6.0),
        "flip": rng.random() < 0.5,
        "brightness": 1.0 + rng.uniform(-0.15, 0.15),
        "contrast": 1.0 + rng.uniform(-0.15, 0.15),
        "color": 1.0 + rng.uniform(-0.15, 0.15),
        "blur_radius": rng.choice([0.0, 0.0, 0.4, 0.8]),
    }


def apply_consistent_clip_transform(image, params, output_size):
    image = image.convert("RGB")
    width, height = image.size

    crop_scale = params["crop_scale"]
    crop_width = max(8, int(width * crop_scale))
    crop_height = max(8, int(height * crop_scale))
    max_left = max(0, width - crop_width)
    max_top = max(0, height - crop_height)
    left = int(round(max_left * params["x_shift"]))
    top = int(round(max_top * params["y_shift"]))

    image = image.crop((left, top, left + crop_width, top + crop_height))
    image = image.resize((output_size, output_size), Image.BILINEAR)

    if params["flip"]:
        image = ImageOps.mirror(image)

    if abs(params["rotation"]) > 0.1:
        image = image.rotate(params["rotation"], resample=Image.BILINEAR)

    image = ImageEnhance.Brightness(image).enhance(params["brightness"])
    image = ImageEnhance.Contrast(image).enhance(params["contrast"])
    image = ImageEnhance.Color(image).enhance(params["color"])

    if params["blur_radius"] > 0:
        image = image.filter(ImageFilter.GaussianBlur(radius=params["blur_radius"]))

    return image


augmented_records = []

if CFG["build_augmented_variants"]:
    augment_candidates = planned_augments_df[planned_augments_df["planned_augmented_variants"] > 0].copy()

    for row in tqdm(augment_candidates.itertuples(index=False), total=len(augment_candidates), desc="augment_clips"):
        source_video_id = str(row.source_video_id)
        source_frame_dir = Path(row.frame_dir)
        frame_paths = sorted_frame_paths(source_frame_dir)
        if not frame_paths:
            continue

        for aug_index in range(1, int(row.planned_augmented_variants) + 1):
            video_key = f"{source_video_id}__aug{aug_index:02d}"
            output_dir = FRAMES_ROOT / video_key

            if output_dir.exists() and not CFG["overwrite_existing_augmented_frames"]:
                output_dir.mkdir(parents=True, exist_ok=True)
            else:
                clear_directory(output_dir)

            params = make_clip_augmentation_params(source_video_id, aug_index, CFG["seed"])

            for frame_number, frame_path in enumerate(frame_paths):
                with Image.open(frame_path) as image:
                    transformed = apply_consistent_clip_transform(
                        image=image,
                        params=params,
                        output_size=CFG["extract_size"],
                    )
                    transformed.save(
                        output_dir / f"{frame_number:04d}.jpg",
                        quality=CFG["jpeg_quality"],
                        optimize=True,
                    )

            record = {
                "video_key": video_key,
                "source_video_id": source_video_id,
                "video_path": source_frame_dir.as_posix(),
                "frame_dir": str(output_dir),
                "is_augmented": 1,
                "aug_index": aug_index,
                "frame_count_extracted": len(frame_paths),
                "fps": np.nan,
                "duration_sec": np.nan,
                "width": CFG["extract_size"],
                "height": CFG["extract_size"],
            }
            for label in label_columns:
                record[label] = int(getattr(row, label))
            augmented_records.append(record)

augmented_manifest_df = pd.concat(
    [base_manifest_df, pd.DataFrame(augmented_records)],
    ignore_index=True,
)
augmented_manifest_df.to_csv(AUGMENTED_MANIFEST_PATH, index=False)

print(f"augmented rows added: {len(augmented_records):,}")
print(f"combined manifest rows: {len(augmented_manifest_df):,}")
print("combined manifest:", AUGMENTED_MANIFEST_PATH)

trainable_balance = pd.DataFrame(
    {
        "base_positives": base_manifest_df[label_columns].sum().astype(int),
        "after_offline_aug": augmented_manifest_df[label_columns].sum().astype(int),
    }
)
trainable_balance["base_to_aug_ratio"] = (
    trainable_balance["after_offline_aug"]
    / trainable_balance["base_positives"].clip(lower=1)
).round(2)

display(trainable_balance.sort_values("after_offline_aug", ascending=False))
display(augmented_manifest_df.head())


augment_clips: 100%|████████████████████████████████████████████████████████████████████████████| 917/917 [04:10<00:00,  3.67it/s]

augmented rows added: 1,566
combined manifest rows: 6,684
combined manifest: E:\m-hvc\datasets\data-from-juniors\full_dataset_augmented_manifest.csv


,base_positives,after_offline_aug,base_to_aug_ratio
humour,4396,5278,1.20
sensitive,700,971,1.39
derogatory__lang,351,459,1.31
anger,373,448,1.20
generic,318,424,1.33
positive,190,412,2.17
threat,149,340,2.28
emotional,164,334,2.04
gender_hate,109,331,3.04
political_hate,131,320,2.44


,video_key,source_video_id,video_path,frame_dir,is_augmented,aug_index,frame_count_extracted,fps,duration_sec,width,...,political_hate,religion_hate,informative,ethinity_hate,anger,emotional,social_hate,controversial,indv_hate,gender_hate
0,2,2,E:\m-hvc\datasets\data-from-juniors\videos\2.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
1,9,9,E:\m-hvc\datasets\data-from-juniors\videos\9.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
2,10,10,E:\m-hvc\datasets\data-from-juniors\videos\10.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
3,11,11,E:\m-hvc\datasets\data-from-juniors\videos\11.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,E:\m-hvc\datasets\data-from-juniors\videos\1.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
